In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from fraud_detection.data.load import load_interim
from fraud_detection.data.split import assign_split

In [ ]:
df = load_interim()
df["split"] = assign_split(df)
df.groupby("split")["isFraud"].agg(["size", "mean"])

In [ ]:
df["day"] = df["TransactionDT"] // 86400
key_cols = ["card1", "card2", "card3", "card5", "card6", "addr1"]
uid = (
    df[key_cols].astype("string").fillna("NA").agg("_".join, axis=1)
    + "_"
    + (df["day"] - df["D1"]).astype("string").fillna("NA")
)

train_clients = set(uid[df["split"] == "train"])
seen = uid[df["split"] == "test"].isin(train_clients)
print(f"{seen.mean():.0%} des transactions de test viennent de clients déjà vus")
print(df.loc[df["split"] == "test"].groupby(seen.values)["isFraud"].mean())